# CARDIODIAG: Generalization-Aware and Uncertainty-Calibrated Cardiovascular Disease Prediction

This notebook is self-contained and runs on Kaggle CPU. It implements the corrected selector-boundary protocol, leakage-safe nested validation, fixed OOF ensemble comparison, calibration, reliability, selective prediction, split-conformal prediction, and feature-selection stability. It never claims external validation or clinical readiness.

## 1. Runtime and mode
Set `RUN_MODE` to `"smoke"` for a fast validation run or `"full"` for the larger prespecified run. No package or project module outside this notebook is required. Internet is not needed.

In [ ]:
# Optional installation cell. Leave INSTALL_DEPENDENCIES=False on Kaggle when the standard image already has these packages.
import sys, subprocess, importlib.util, platform, os, json, hashlib, time, warnings
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy", "pandas", "scipy", "scikit-learn", "matplotlib"])
required = ["numpy", "pandas", "scipy", "sklearn", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
print("Python", platform.python_version(), "missing", missing)
if missing:
    raise ImportError("Missing packages: " + ", ".join(missing) + ". Enable the optional install cell or use a standard Kaggle Python image.")
RUN_MODE = "smoke"  # change to "full" for the larger run
SEED = 42
CONFIG = {
    "run_mode": RUN_MODE, "seed": SEED,
    "outer_splits": 3 if RUN_MODE == "smoke" else 5,
    "inner_splits": 2 if RUN_MODE == "smoke" else 3,
    "selector_resamples": 3 if RUN_MODE == "smoke" else 8,
    "stability_repeats": 3 if RUN_MODE == "smoke" else 8,
    "bootstrap_interval_repetitions": 50 if RUN_MODE == "smoke" else 200,
    "primary_metric": "roc_auc", "threshold": 0.5,
    "selector_protocol": "post_boundary_correction_v1",
}
CONFIG

## 2. Imports and self-contained implementations

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit, logit
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, brier_score_loss, log_loss, confusion_matrix, roc_curve, precision_recall_curve)
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.utils.validation import check_array, check_X_y, check_is_fitted

warnings.filterwarnings("ignore", category=FutureWarning)
PROJECT_ROOT = Path.cwd()
RESULT_DIR = Path("/kaggle/working/CARDIODIAG_results") if Path("/kaggle/working").exists() else PROJECT_ROOT / "CARDIODIAG_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def discover(filename):
    candidates = [PROJECT_ROOT / filename, PROJECT_ROOT / "data" / filename, RESULT_DIR / filename]
    if Path("/kaggle/input").exists(): candidates += list(Path("/kaggle/input").glob("**/" + filename))
    return next((p for p in candidates if p.exists()), None)

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

def load_dataset(path, target):
    sep = ";" if path.name == "cardio_train.csv" else ","
    df = pd.read_csv(path, sep=sep)
    if target not in df: raise ValueError(f"Target {target!r} missing from {path}")
    y = pd.to_numeric(df.pop(target), errors="raise").astype(int)
    if set(y.unique()) != {0,1}: raise ValueError(f"{path}: target must be binary 0/1")
    return df, y

def drop_identifiers(X):
    names=[]
    for c in list(X.columns):
        low=c.lower()
        if low in {"id","patientid","patient_id","unnamed: 0"} or low.endswith("_id"):
            names.append(c)
    return X.drop(columns=names), names

def dataset_summary(path,target):
    X,y=load_dataset(path,target); Xm,dropped=drop_identifiers(X)
    nums=Xm.select_dtypes(include=[np.number])
    return {"path":str(path),"rows":len(X),"columns":list(X.columns),"target":target,"target_counts":{str(k):int(v) for k,v in y.value_counts().sort_index().items()},"identifier_columns":dropped,"duplicate_rows":int(pd.concat([X,y.rename(target)],axis=1).duplicated().sum()),"missing_values":int(X.isna().sum().sum()),"sha256":sha256(path),"finite_numeric":bool(np.isfinite(nums.to_numpy(dtype=float)).all())}

def make_preprocessor(X):
    cat=[c for c in X.columns if X[c].dtype == "object" or str(X[c].dtype).startswith("category") or (X[c].nunique(dropna=True) <= 8 and c.lower() in {"gender","chestpain","cholesterol","gluc","smoke","alco","active"})]
    num=[c for c in X.columns if c not in cat]
    pre=ColumnTransformer([("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),cat)],remainder="drop")
    return pre, num, cat

class StableXAIHDRC(BaseEstimator, TransformerMixin):
    """Corrected selector: split original positions, bootstrap fit positions only, validate relevance on untouched positions."""
    def __init__(self,n_resamples=8,stability_threshold=0.75,relevance_threshold=0.02,fallback_top_k=8,random_state=42,relevance_validation_fraction=0.25):
        self.n_resamples=n_resamples; self.stability_threshold=stability_threshold; self.relevance_threshold=relevance_threshold; self.fallback_top_k=fallback_top_k; self.random_state=random_state; self.relevance_validation_fraction=relevance_validation_fraction
    def fit(self,X,y):
        X,y=check_X_y(X,y,accept_sparse=False,dtype=float)
        if np.unique(y).size != 2: raise ValueError("StableXAIHDRC supports binary targets only")
        original=np.arange(len(y))
        fit_idx,val_idx=train_test_split(original,test_size=self.relevance_validation_fraction,stratify=y,random_state=self.random_state)
        self.relevance_fit_indices_=fit_idx; self.relevance_validation_indices_=val_idx
        rng=np.random.default_rng(self.random_state); selections=[]; relevances=[]; boots=[]
        class_indices=[fit_idx[y[fit_idx]==label] for label in np.unique(y)]
        for _ in range(self.n_resamples):
            sampled=np.concatenate([rng.choice(ci,size=len(ci),replace=True) for ci in class_indices]); rng.shuffle(sampled); boots.append(sampled)
            model=LogisticRegression(max_iter=1000,class_weight="balanced",random_state=int(rng.integers(2**31-1))).fit(X[sampled],y[sampled])
            corr=np.abs(np.corrcoef(X[val_idx].T,y[val_idx])[0:-1,-1]); corr=np.nan_to_num(corr)
            perm=permutation_importance(model,X[val_idx],y[val_idx],scoring="roc_auc",n_repeats=2,random_state=int(rng.integers(2**31-1))).importances_mean; perm=np.maximum(perm,0); perm=perm/perm.max() if perm.max()>0 else perm
            rel=0.5*corr+0.5*perm; relevances.append(rel); selections.append(rel >= self.relevance_threshold)
        self.bootstrap_indices_=tuple(boots); self.selection_matrix_=np.asarray(selections,dtype=float); self.selection_frequency_=self.selection_matrix_.mean(0); self.mean_relevance_=np.asarray(relevances).mean(0)
        support=(self.selection_frequency_>=self.stability_threshold)&(self.mean_relevance_>=self.relevance_threshold)
        if not support.any(): support[np.argsort(self.mean_relevance_)[::-1][:min(self.fallback_top_k,X.shape[1])]]=True
        self.support_=support; self.selected_indices_=np.flatnonzero(support); self.n_features_in_=X.shape[1]; return self
    def transform(self,X):
        check_is_fitted(self,["support_"]); X=check_array(X,accept_sparse=False,dtype=float); return X[:,self.selected_indices_]
    def get_support(self,indices=False):
        check_is_fitted(self,["support_"]); return self.selected_indices_ if indices else self.support_

def ece(y,p,bins=10):
    y=np.asarray(y); p=np.asarray(p); edges=np.linspace(0,1,bins+1); total=0.0
    for lo,hi in zip(edges[:-1],edges[1:]):
        mask=(p>=lo)&(p<hi if hi<1 else p<=hi)
        if mask.any(): total += mask.mean()*abs(p[mask].mean()-y[mask].mean())
    return float(total)

def metrics(y,p):
    y=np.asarray(y); p=np.asarray(p); pred=(p>=CONFIG["threshold"]).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {"roc_auc":float(roc_auc_score(y,p)),"pr_auc":float(average_precision_score(y,p)),"accuracy":float(accuracy_score(y,pred)),"balanced_accuracy":float(balanced_accuracy_score(y,pred)),"sensitivity":float(recall_score(y,pred,zero_division=0)),"specificity":float(tn/(tn+fp)) if tn+fp else float("nan"),"precision":float(precision_score(y,pred,zero_division=0)),"f1":float(f1_score(y,pred,zero_division=0)),"brier":float(brier_score_loss(y,p)),"log_loss":float(log_loss(y,p,labels=[0,1])),"ece":ece(y,p),"confusion_matrix":[[int(tn),int(fp)],[int(fn),int(tp)]]}

def nogueira(M):
    M=np.asarray(M,dtype=float); n,p=M.shape; k=M.sum(1).mean(); denom=p*(k/p)*(1-k/p); return float(1-(np.mean(M.var(0)))/(denom/n)) if denom>0 else 1.0

def bootstrap_auc(y,p,reps):
    rng=np.random.default_rng(SEED); vals=[]
    for _ in range(reps):
        ix=rng.integers(0,len(y),len(y));
        if len(np.unique(y[ix]))==2: vals.append(roc_auc_score(y[ix],p[ix]))
    return {"lower":float(np.quantile(vals,.025)),"upper":float(np.quantile(vals,.975)),"repetitions":len(vals),"interpretation":"descriptive row-resampling interval, not independent-patient uncertainty"}

def selective(y,p):
    conf=np.maximum(p,1-p); correct=((p>=.5).astype(int)==y); order=np.argsort(-conf); return [{"coverage":float((i+1)/len(y)),"risk":float(1-correct[order[:i+1]].mean())} for i in np.linspace(0,len(y)-1,min(20,len(y)),dtype=int)]

def conformal_summary(y_cal,p_cal,y_eval,p_eval,alpha=.1):
    cal_scores=1-np.where(y_cal==1,p_cal,1-p_cal); q=float(np.quantile(cal_scores,min(1,(len(cal_scores)+1)*(1-alpha)/len(cal_scores)),method="higher")); sets=np.column_stack([p_eval<=q,1-p_eval<=q]); covered=sets[np.arange(len(y_eval)),y_eval.astype(int)]; singleton=sets.sum(1)==1; pred=np.where(p_eval>=.5,1,0); return {"coverage":float(covered.mean()),"singleton_coverage":float(covered[singleton].mean()) if singleton.any() else float("nan"),"abstention_rate":float((~singleton).mean()),"selective_accuracy":float((pred[singleton]==y_eval[singleton]).mean()) if singleton.any() else float("nan"),"threshold":q}

def provenance(path,target,experiment):
    return {"dataset":str(path),"dataset_sha256":sha256(path),"target":target,"experiment_type":experiment,"python":platform.python_version(),"configuration":CONFIG,"selector_protocol":CONFIG["selector_protocol"]}


## 3. Dataset discovery and validation
The bundled Mendeley file is used when present. Optional heart5 and Kaggle 70k datasets are never substituted and are reported as skipped when absent.

In [ ]:
DATASETS={"mendeley":(discover("MendeleyCardiovascular_Disease_Dataset.csv"),"target"),"heart5":(discover("heart5.csv"),"HeartDisease"),"kaggle70k":(discover("cardio_train.csv"),"cardio")}
AVAILABLE={k:v for k,v in DATASETS.items() if v[0] is not None}
SKIPPED={k:"dataset not attached; see README.txt" for k,v in DATASETS.items() if v[0] is None}
if not AVAILABLE: raise FileNotFoundError("Attach at least one permitted dataset.")
SUMMARIES={k:dataset_summary(p,t) for k,(p,t) in AVAILABLE.items()}
print("Available:",list(AVAILABLE)); print("Skipped:",SKIPPED); print(json.dumps(SUMMARIES,indent=2))


## 4. Leakage-safe nested validation
Preprocessing is fitted inside every inner/outer training fold. Hyperparameters are selected only by inner ROC-AUC. Outer-test observations are used only for final prediction scoring.

In [ ]:
def nested_dataset(path,target):
    X,y=load_dataset(path,target); X,dropped=drop_identifiers(X); outer=StratifiedKFold(CONFIG["outer_splits"],shuffle=True,random_state=SEED)
    specs={"logistic":(LogisticRegression(max_iter=1500,class_weight="balanced",random_state=SEED),{"model__C":[1.0] if RUN_MODE=="smoke" else {"model__C":[.1,1,10]}["model__C"]}),"random_forest":(RandomForestClassifier(n_estimators=80 if RUN_MODE=="smoke" else 200,n_jobs=1,class_weight="balanced",random_state=SEED),{"model__max_depth":[None,16]}),"hist_gradient_boosting":(HistGradientBoostingClassifier(max_iter=80 if RUN_MODE=="smoke" else 180,random_state=SEED),{"model__max_leaf_nodes":[15,31]})}
    result={"path":str(path),"target":target,"dropped_identifiers":dropped,"models":{},"configuration":CONFIG,"provenance":provenance(path,target,"nested_validation")}
    for name,(model,grid) in specs.items():
        p=np.full(len(y),np.nan); folds=[]
        for fold,(tr,te) in enumerate(outer.split(X,y)):
            pre,_,_=make_preprocessor(X.iloc[tr]); pipe=Pipeline([("preprocessor",pre),("model",clone(model))]); search=GridSearchCV(pipe,grid,scoring="roc_auc",cv=CONFIG["inner_splits"],n_jobs=1,refit=True); search.fit(X.iloc[tr],y.iloc[tr]); p[te]=search.predict_proba(X.iloc[te])[:,1]; folds.append({"fold":fold,"train_rows":len(tr),"test_rows":len(te),"best_params":search.best_params_,"metrics":metrics(y.iloc[te],p[te])})
        result["models"][name]={"pooled":metrics(y,p),"folds":folds,"oof_probabilities":p.tolist(),"descriptive_auc_interval":bootstrap_auc(y.to_numpy(),p,CONFIG["bootstrap_interval_repetitions"])}
    ensemble=np.mean([np.asarray(v["oof_probabilities"]) for v in result["models"].values()],axis=0); result["fixed_soft_voting"]={"weights":"equal","metrics":metrics(y,ensemble)}
    return result

RESULTS={}
for name,(path,target) in AVAILABLE.items():
    t=time.perf_counter(); RESULTS[name]=nested_dataset(path,target); RESULTS[name]["runtime_seconds"]=time.perf_counter()-t
    print(name,{m:v["pooled"]["roc_auc"] for m,v in RESULTS[name]["models"].items()},"ensemble",RESULTS[name]["fixed_soft_voting"]["metrics"]["roc_auc"])


## 5. Separate calibration, reliability, uncertainty, and conformal analysis
Calibration observations are separate from model-fitting and evaluation observations. Conformal scores use a separate calibration partition. These are internal validation results, not external validation.

In [ ]:
ANALYSIS={}
for name,(path,target) in AVAILABLE.items():
    X,y=load_dataset(path,target); X,_=drop_identifiers(X); tr,rest=train_test_split(np.arange(len(y)),test_size=.4,stratify=y,random_state=SEED); cal,ev=train_test_split(rest,test_size=.5,stratify=y.iloc[rest],random_state=SEED)
    pre,_,_=make_preprocessor(X.iloc[tr]); model=Pipeline([("preprocessor",pre),("model",HistGradientBoostingClassifier(max_iter=100,random_state=SEED))]).fit(X.iloc[tr],y.iloc[tr]); pcal=model.predict_proba(X.iloc[cal])[:,1]; pev=model.predict_proba(X.iloc[ev])[:,1]
    calibrator=LogisticRegression(random_state=SEED).fit(logit(np.clip(pcal,1e-6,1-1e-6)).reshape(-1,1),y.iloc[cal]); pc=calibrator.predict_proba(logit(np.clip(pev,1e-6,1-1e-6)).reshape(-1,1))[:,1]; ye=y.iloc[ev].to_numpy()
    # Corrected selector is fitted only on the training partition after preprocessing.
    pre2,_,_=make_preprocessor(X.iloc[tr]); Xt=pre2.fit_transform(X.iloc[tr]); selector=StableXAIHDRC(n_resamples=CONFIG["selector_resamples"],random_state=SEED).fit(Xt,y.iloc[tr]); selected_model=LogisticRegression(max_iter=1500,class_weight="balanced",random_state=SEED).fit(selector.transform(Xt),y.iloc[tr]); test_trans=pre2.transform(X.iloc[ev]); ps=selected_model.predict_proba(selector.transform(test_trans))[:,1]
    stability=[]
    for r in range(CONFIG["stability_repeats"]):
        ix=np.random.default_rng(SEED+r).choice(len(tr),len(tr),replace=True); s=StableXAIHDRC(n_resamples=CONFIG["selector_resamples"],random_state=SEED+r).fit(Xt[ix],y.iloc[tr].to_numpy()[ix]); stability.append(s.get_support().astype(int))
    ANALYSIS[name]={"partitions":{"fit_rows":len(tr),"calibration_rows":len(cal),"evaluation_rows":len(ev)},"uncalibrated_metrics":metrics(ye,pev),"calibrated_metrics":metrics(ye,pc),"ece":ece(ye,pc),"selective_risk":selective(ye,pc),"full_coverage_risk":float(np.mean((pc>=.5)!=ye)),"conformal":conformal_summary(y.iloc[cal].to_numpy(),pcal,ye,pev),"corrected_selector":{"protocol":CONFIG["selector_protocol"],"selected_features":int(selector.get_support().sum()),"fit_positions":selector.relevance_fit_indices_.tolist(),"validation_positions":selector.relevance_validation_indices_.tolist(),"bootstrap_positions_within_fit":bool(all(np.isin(b,selector.relevance_fit_indices_).all() for b in selector.bootstrap_indices_)),"test_metrics":metrics(ye,ps),"nogueira_stability":nogueira(np.asarray(stability))}}
    print(name,ANALYSIS[name]["calibrated_metrics"],ANALYSIS[name]["conformal"])


## 6. Diagnostic plots and saved artifacts

In [ ]:
FIG_DIR=RESULT_DIR/"figures"; FIG_DIR.mkdir(exist_ok=True)
for name,res in RESULTS.items():
    path,target=AVAILABLE[name]; y=load_dataset(path,target)[1].to_numpy(); fig,ax=plt.subplots(1,3,figsize=(15,4))
    for model,v in res["models"].items():
        p=np.asarray(v["oof_probabilities"]); fpr,tpr,_=roc_curve(y,p); ax[0].plot(fpr,tpr,label=f"{model} ({v['pooled']['roc_auc']:.3f})"); pr,rc,_=precision_recall_curve(y,p); ax[1].plot(rc,pr,label=model); ax[2].plot(np.sort(p),np.sort(y),".",alpha=.2,label=model)
    ax[0].plot([0,1],[0,1],"k--"); ax[0].set_title("ROC"); ax[1].set_title("Precision-recall"); ax[2].set_title("Sorted probabilities/labels"); [a.legend() for a in ax]; fig.tight_layout(); fig.savefig(FIG_DIR/f"{name}_diagnostics.png",dpi=120); plt.close(fig)
OUTPUT={"configuration":CONFIG,"datasets":SUMMARIES,"skipped_datasets":SKIPPED,"nested_validation":RESULTS,"reliability_analysis":ANALYSIS,"limitations":["Each dataset is reported separately; no pooled estimate.","Descriptive bootstrap intervals resample observed rows and are not independent-patient uncertainty.","Missing optional datasets are skipped rather than substituted.","No external validation, clinical utility, deployment readiness, or universal generalization claim."],"provenance":{"notebook":"CARDIODIAG_Kaggle.ipynb","python":platform.python_version()}}
(RESULT_DIR/"CARDIODIAG_metrics.json").write_text(json.dumps(OUTPUT,indent=2,default=str),encoding="utf-8")
(RESULT_DIR/"CARDIODIAG_config.json").write_text(json.dumps(CONFIG,indent=2),encoding="utf-8")
print("Saved",RESULT_DIR/"CARDIODIAG_metrics.json", "and",len(list(FIG_DIR.glob("*.png"))),"figures")


## 7. Interpretation
The notebook reports model and uncertainty metrics without selecting a result after seeing the outer-test labels. High performance on a small or easy dataset is not evidence of external generalization. Review duplicate counts, target semantics, calibration, conformal coverage, and split variability before making claims.